# Mission 02 - FST Dialogue Systems, Lovelace's Objection & Noisy Channels

**Squad:** Goodfellow (Group 18) | **Role:** Core Architect / Systems Architect / Lead Engineer | **Author:** Abdul Jalil Memon

This notebook drives `04_fst_noisy_simulator.py`, which holds all the logic:

| Part | What it shows |
|---|---|
| 1 | The FST dialogue manager walking its states and filling slots |
| 2 | The noisy channel: word counter, noise-token injector (append / substitute), entropy |
| 3 | The five stress tests (T1-T5) against the FST |
| 4 | The live 10-turn Anti-Turing interrogation with Gemini through the channel |
| 5 | Saving `05_Noisy_Interrogation_Transcript.md` |

**Before running:** put `GEMINI_API_KEY` (and optionally `GEMINI_MODEL`) in a local `.env` file. Never commit it.
Use *Restart Kernel and Run All* so every output is fresh.

## 0. Setup

In [1]:
import importlib.util, os, sys
from pathlib import Path

HERE = Path.cwd()  # run the notebook from the folder that contains 04_fst_noisy_simulator.py
spec = importlib.util.spec_from_file_location("sim", HERE / "04_fst_noisy_simulator.py")
sim = importlib.util.module_from_spec(spec)
sys.modules["sim"] = sim
spec.loader.exec_module(sim)
sim.load_dotenv()

FST = sim.load_fst_class()      # baseline class, or the squad's final class via FST_MODULE
SEED = 42
NOISE_MODE = "append"           # "append" (6 words + noise token) or "substitute" (7 words, one replaced)
HISTORY = "raw"                 # what the model sees of its own earlier replies: "raw" or "noisy"
USE_PERSONA = True             # False = send the prompts with no system prompt (full-length answers, baseline)
SUFFIX = ""                    # e.g. "_baseline" so a second run does not overwrite the first transcript
PROVIDER = os.environ.get("NOTEBOOK_PROVIDER", "auto")

print("FST class:", FST.__module__ + "." + FST.__name__)
print("Noise vocabulary:", sim.NOISE_VOCABULARY)

FST class: sim.FSTDialogueManager
Noise vocabulary: ['Banana', 'Labyrinth', 'Quantum', 'Midnight', 'Velocity', 'Green', 'Cosmological']


## 1. FST dialogue manager

Happy path: `INIT -> GREETING -> COLLECT_SLOTS -> CONFIRMATION -> TERMINAL`, with the slots `destination`, `speed` and `clearance` filled in order.

In [2]:
fst = FST()
crew_inputs = ["Hello AURA", "Proxima Centauri", "Warp 7", "CLEARANCE-ALPHA-9", "YES"]
print(f"{'#':<3} {'input':<20} {'before':<14} {'after':<14} slots")
print("-" * 90)
for i, text in enumerate(crew_inputs, 1):
    before = fst.state
    reply = fst.process_utterance(text)
    print(f"{i:<3} {text:<20} {before:<14} {fst.state:<14} {fst.slot_memory}")
    print(f"    AURA: {reply}")

#   input                before         after          slots
------------------------------------------------------------------------------------------
1   Hello AURA           INIT           GREETING       {}
    AURA: AURA FST Online. State: GREETING. Welcome aboard USSE Noosphere. What is your destination?
2   Proxima Centauri     GREETING       COLLECT_SLOTS  {'destination': 'Proxima Centauri'}
    AURA: Destination: 'Proxima Centauri'. State: COLLECT_SLOTS. Please specify speed (Warp 1-9):
3   Warp 7               COLLECT_SLOTS  COLLECT_SLOTS  {'destination': 'Proxima Centauri', 'speed': 'Warp 7'}
    AURA: Speed: 'Warp 7' recorded. Please provide your security clearance code:
4   CLEARANCE-ALPHA-9    COLLECT_SLOTS  CONFIRMATION   {'destination': 'Proxima Centauri', 'speed': 'Warp 7', 'clearance': 'CLEARANCE-ALPHA-9'}
    AURA: [CONFIRM] Course to 'Proxima Centauri' at 'Warp 7'. Confirm? (YES / NO)
5   YES                  CONFIRMATION   TERMINAL       {'destination': 'Proxima Cen

## 2. Noisy channel

Every FST reply is cut to the 7-word budget and carries one noise token. A "word" is a whitespace-delimited token (`str.split()`).

In [3]:
sample = "I am operating normally under signal degradation right now"
for mode in ("append", "substitute"):
    ch = sim.NoisyChannelSimulator("demo", seed=SEED)
    out = ch.transmit(sample, mode)
    decoded, ok = ch.verify_transmission(out, ch.last_noise_index)
    loss = ch.analyze_semantic_loss(sample, out, ch.last_noise_index)
    print(f"[{mode}]")
    print(f"  raw ({ch.count_words(sample)} words): {sample}")
    print(f"  sent ({ch.count_words(out)} words): {out}")
    print(f"  noise token: {loss['injected_sentinel']} at index {ch.last_noise_index}, valid={ok}")
    print(f"  words kept: {loss['semantic_words_retained']}, cut: {loss['words_truncated']}")

[append]
  raw (9 words): I am operating normally under signal degradation right now
  sent (7 words): I am operating normally under signal Green
  noise token: Green at index 6, valid=True
  words kept: 6, cut: 3
[substitute]
  raw (9 words): I am operating normally under signal degradation right now
  sent (7 words): Green am operating normally under signal degradation
  noise token: Green at index 0, valid=True
  words kept: 6, cut: 3


In [4]:
# Entropy of the noise source over many draws: it approaches log2(7) = 2.8074 bits.
ch = sim.NoisyChannelSimulator("entropy-demo", seed=SEED)
for n in (10, 100, 1000, 10000):
    ch.reset()
    for _ in range(n):
        ch.transmit(sample, NOISE_MODE)
    print(f"n={n:>5}  H(noise) = {ch.channel_entropy():.4f} bits  (max {ch.max_entropy():.4f})")

n=   10  H(noise) = 1.8955 bits  (max 2.8074)
n=  100  H(noise) = 2.7742 bits  (max 2.8074)
n= 1000  H(noise) = 2.8056 bits  (max 2.8074)
n=10000  H(noise) = 2.8068 bits  (max 2.8074)


## 3. Stress tests T1-T5

Each test replays a prefix of the dialogue, then sends one non-canonical input. `hazard_reproduced = True` means the FST under test still shows the documented failure; `False` means it behaves differently (for example, it was fixed).

In [5]:
results = sim.run_stress_test_suite(FST)
print(f"{'Test':<5} {'Name':<24} {'Input':<30} {'Final state':<14} {'Hazard':<9} Severity")
print("-" * 100)
for r in results:
    print(f"{r['test_id']:<5} {r['name']:<24} {r['trigger_input']:<30} {r['final_state']:<14} "
          f"{str(r['hazard_reproduced']):<9} {r['severity']}")
print()
for r in results:
    print(f"{r['test_id']}: {r['hazard']}  slots={r['recorded_slots']}")

Test  Name                     Input                          Final state    Hazard    Severity
----------------------------------------------------------------------------------------------------
T1    Off-topic destination    What is the weather?           COLLECT_SLOTS  True      Medium
T2    Invalid speed            Warp 99                        COLLECT_SLOTS  True      High
T3    Sarcastic confirmation   Yes, obviously not!            TERMINAL       True      Critical
T4    Slot confusion           CLEARANCE-ALPHA-9              COLLECT_SLOTS  True      High
T5    Ambiguous refusal        No, please change destination  FALLBACK       True      Low-Medium

T1: Off-topic utterance accepted as destination.  slots={'destination': 'What is the weather?'}
T2: Out-of-range speed accepted without validation.  slots={'destination': 'Proxima Centauri', 'speed': 'Warp 99'}
T3: 'yes' matched despite negation; course locked.  slots={'destination': 'Proxima Centauri', 'speed': 'Warp 7', 'clear

## 4. Live interrogation (10 turns, one conversation)

The ten prompts come from the squad's question bank (Abubaker's five questions plus follow-ups). Each raw reply from the model is passed through the noisy channel; the ledger below shows what the crew would receive. All turns are written to `logs/run_*.jsonl`.

In [ ]:
def resolve_provider(name):
    if name != "auto":
        return name
    if os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"):
        return "gemini"
    if os.environ.get("OPENAI_API_KEY"):
        return "openai"
    if os.environ.get("ANTHROPIC_API_KEY"):
        return "anthropic"
    return "mock"

provider = resolve_provider(PROVIDER)
llm = sim.LLMClient(provider, use_system_prompt=USE_PERSONA, noise_mode=NOISE_MODE)
channel = sim.NoisyChannelSimulator("Sirius B Transceiver", seed=SEED)
logger = sim.RunLogger(HERE / "logs")
runner = sim.AntiTuringInterrogationRunner(llm, channel, logger)

print("Provider:", provider, "| model:", llm.model, "| noise:", NOISE_MODE, "| history:", HISTORY, "| seed:", SEED)
if provider == "mock":
    print("WARNING: mock provider. Output is canned test text, not model output. Set GEMINI_API_KEY.")

logger.log_event("run_metadata", {"noise_mode": NOISE_MODE, "history": HISTORY, "seed": SEED,
                                  "fst_class": FST.__module__ + "." + FST.__name__, "llm": llm.settings(),
                                  "source": "ps02_solution.ipynb"})
turns = runner.run_all_turns(NOISE_MODE, HISTORY)

print(f"{'Turn':<5} {'ID':<13} {'Raw':<5} {'Kept':<6} Transmitted")
print("-" * 80)
for t in turns:
    print(f"{t['turn']:<5} {t['question_id']:<13} {t['raw_word_count']:<5} "
          f"{t['semantic_retention_rate'] * 100:>3.0f}%   {t['noisy_transmitted_message']}")
print("\nLog file:", logger.log_path)

In [ ]:
print(channel.noise_token_report())

In [ ]:
# Per-turn view: how much of each reply survives, and the running noise entropy.
try:
    import matplotlib.pyplot as plt
    ids = [t["question_id"] for t in turns]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    ax[0].bar(ids, [t["semantic_retention_rate"] * 100 for t in turns])
    ax[0].set_ylabel("% of raw reply words kept"); ax[0].set_title("Semantic retention per turn")
    ax[0].tick_params(axis="x", rotation=60)
    ax[1].plot(range(1, len(turns) + 1), [t["cumulative_entropy_bits"] for t in turns], marker="o")
    ax[1].axhline(channel.max_entropy(), linestyle="--")
    ax[1].set_xlabel("turn"); ax[1].set_ylabel("bits"); ax[1].set_title("Cumulative noise entropy (dashed: log2 7)")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed; skipping plots (pip install matplotlib).")

In [ ]:
# Heuristic flags computed on the RAW replies (not ratings; rate them yourself from the raw text).
for t in turns:
    if t["auto_checks"]:
        print(t["question_id"], t["auto_checks"])

## 5. Save the transcript

In [ ]:
fname = f"05_Noisy_Interrogation_Transcript{SUFFIX}" + ("_MOCK.md" if provider == "mock" else ".md")
out_path = sim.save_transcript(
    sim.SCRIPT_DIR / fname,
    runner.format_markdown_transcript({"noise_mode": NOISE_MODE, "history_mode": HISTORY}))
print("Saved (UTF-8; existing files preserved):", out_path)


## 6. Evaluation and execution status

The saved live transcript contains an AI-assisted assessment in Section 5, pending Jalil’s review. See `05_Noisy_Interrogation_Transcript.md`. Its 94.0% word retention is not a measure of semantic accuracy.

During verification, only setup, FST, channel and stress-test cells were executed offline. Live interrogation, plotting and transcript-export cells were not rerun; their execution counts remain empty. Jalil should run the live section and review its evaluation before sign-off. Exports preserve existing files automatically and use _run2, _run3, etc. for later runs. Each new run requires its own assessment. Restart the kernel before rerunning to load the updated export helper.
